# Análisis de la Relación entre Ocupación Hospitalaria y Letalidad
## en Establecimientos Hospitalarios Públicos de Chile (2014-2025)

**Segundo Avance de Proyecto**

| | |
|---|---|
| **Integrante** | Benjamín Bravo |
| **Profesor/a** | Cristian García |
| **Ayudante** | Benjamín Bennett |
| **Curso** | Análisis de Datos e Inferencia Estadística |
| **Fecha** | Mayo 2026 |

---

## Introducción

El sistema hospitalario público de Chile opera con apenas 2 camas por cada mil habitantes, muy por debajo del promedio de la OCDE. Eso genera niveles de ocupación estructuralmente altos. La literatura internacional indica que tasas superiores al 85% se asocian a deterioro en la calidad de atención, pero no está claro si eso se traduce en mayor mortalidad en el contexto chileno.

Este cuaderno analiza datos del REM20 (DEIS/MINSAL) para el período 2014-2025 y busca responder la siguiente pregunta:

**¿Existe una relación estadísticamente significativa entre el índice de ocupación hospitalaria y la letalidad en los establecimientos hospitalarios públicos de Chile?**

**Hipótesis:** A mayor índice de ocupación, mayor es la letalidad promedio. Esta relación es más pronunciada en áreas de alta complejidad clínica.

El análisis sigue este orden:
1. Carga e inspección inicial
2. Clasificación de variables
3. Limpieza de datos
4. Análisis exploratorio (EDA)
5. Test de hipótesis
6. Modelo de regresión lineal múltiple
7. Discusión y conclusiones

## 1. Carga de Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats as scipy_stats
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
pd.set_option('display.max_columns', None)

# Cargamos el dataset REM20. Usamos separador ; porque el archivo viene en formato CSV europeo.
df = pd.read_csv("C:/Users/bravo/OneDrive/Escritorio/Analisis de datos/proyecto_analisis_datos/data/indicadores_rem20_20260225.csv", 
                 sep=';')

# Limpiamos espacios en AREA_FUNCIONAL para evitar errores al hacer filtros y agrupaciones
df["AREA_FUNCIONAL"] = df["AREA_FUNCIONAL"].str.strip()

print(f'Periodo: {df["PERIODO"].min()}-{df["PERIODO"].max()}')
display(df.head())

# Definimos la paleta de colores por complejidad aqui para reutilizarla en todos los graficos
palette = {'Alta complejidad': 'tomato', 'Media complejidad': 'steelblue', 'Baja complejidad': 'seagreen'}

## 2. Inspección Inicial

Revisamos la estructura general del dataset: cuántas filas y columnas tiene, qué tipo de datos tiene cada columna y si hay algo raro a simple vista.

In [ ]:
print(f"Dimensiones del dataset: {df.shape[0]:,} filas | {df.shape[1]} columnas")
print("="*60)
df.info()

## 3. Clasificación de Variables

Clasificamos las variables según su tipo para saber cómo tratar cada una en el análisis. Las variables cuantitativas continuas son las más relevantes para la pregunta de investigación.

In [ ]:
clasificacion = {
    'Identificadores': ['CODIGO_ESTABLECIMIENTO', 'COD_SSS', 'COD_AREA_FUNCIONAL'],
    'Temporales':      ['PERIODO', 'MES'],
    'Categoricas nominales': ['GLOSA_SSS', 'AREA_FUNCIONAL'],
    'Cuantitativas discretas': ['NUMERO_EGRESOS', 'EGRESOS_FALLECIDOS', 'TRASLADOS',
                                 'DIAS_CAMAS_OCUPADAS', 'DIAS_CAMAS_DISPONIBLES', 'DIAS_ESTADA'],
    'Cuantitativas continuas': ['INDICE_OCUPACIONAL', 'PROMEDIO_CAMAS_DISPONIBLE',
                                 'PROMEDIO_DIAS_ESTADA', 'LETALIDAD', 'INDICE_ROTACION']
}

for tipo, cols in clasificacion.items():
    print(f"\n  {tipo} ({len(cols)} variables)")
    print("  " + "-"*40)
    for c in cols:
        if c in df.columns:
            print(f"  - {c}")
        else:
            print(f"  - {c}  no encontrada en el dataset")

## 4. Detección de Valores Nulos y Duplicados

Verificamos que no haya filas vacías ni registros repetidos. Ambos problemas pueden distorsionar el análisis si no se detectan a tiempo.

In [ ]:
# Revisamos cuántos valores nulos hay por columna
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]

if len(nulos) == 0:
    print("Sin valores nulos en el dataset.")
else:
    print("Columnas con nulos:")
    print(nulos)

print()

# Revisamos filas duplicadas exactas
n_dup = df.duplicated().sum()

if n_dup == 0:
    print("Sin filas duplicadas.")
else:
    print(f"Filas duplicadas: {n_dup}")
    display(df[df.duplicated()].head())

## 5. Detección de Problemas de Calidad

Buscamos tres tipos de problemas: valores negativos donde no deberían existir, valores fuera del rango teórico (por ejemplo, un porcentaje mayor a 100%) e inconsistencias lógicas entre columnas relacionadas.

In [ ]:
# Esta funcion recorre cada columna del dataset y busca tres tipos de problemas:
# nulos (valores vacios), negativos (numeros bajo cero donde no deberia haberlos)
# y valores fuera del rango esperado segun el tipo de variable
def detectar_problemas_datos(df):
    problemas = []
    for col in df.columns:
        serie = df[col]
        
        # Contamos cuantos valores vacios tiene la columna
        nulos = serie.isna().sum()
        
        # Solo revisamos negativos en columnas numericas porque en texto no tiene sentido
        negativos = 0
        if pd.api.types.is_numeric_dtype(serie):
            negativos = (serie < 0).sum()
        
        # Para cada columna definimos que rango de valores es valido segun lo que representa
        fuera_rango = 0
        if col == 'MES':
            # El mes solo puede ser entre 1 y 12
            fuera_rango = ((serie < 1) | (serie > 12)).sum()
        elif col == 'PERIODO':
            # El año debe estar dentro del rango razonable del dataset
            fuera_rango = ((serie < 2000) | (serie > 2030)).sum()
        elif col in ['INDICE_OCUPACIONAL', 'LETALIDAD']:
            # Ambas son porcentajes, por lo tanto no pueden ser menores a 0 ni mayores a 100
            fuera_rango = ((serie < 0) | (serie > 100)).sum()
        
        problemas.append({
            'columna': col,
            'nulos': nulos,
            'negativos': negativos,
            'fuera_de_rango': fuera_rango
        })
    return pd.DataFrame(problemas)

# Ejecutamos la funcion y nos quedamos solo con las columnas que tienen algun problema
reporte = detectar_problemas_datos(df)
problemas_reales = reporte[reporte[['nulos','negativos','fuera_de_rango']].sum(axis=1) > 0]

if problemas_reales.empty:
    print("Sin problemas de calidad detectados.")
else:
    print(f"{len(problemas_reales)} columna(s) con problemas:")
    display(problemas_reales)

# Revisamos una inconsistencia logica que no detecta la funcion anterior:
# si un area no tiene camas disponibles, es imposible que tenga camas ocupadas
# estos registros son errores de carga y deben eliminarse
inconsistencias = df[
    (df['DIAS_CAMAS_DISPONIBLES'] == 0) &
    (df['DIAS_CAMAS_OCUPADAS'] > 0)
]
print(f"\nInconsistencias logicas (disponibles=0, ocupadas>0): {len(inconsistencias):,} filas")

## 6. Limpieza del Dataset

Eliminamos los registros con problemas detectados en el paso anterior. Las decisiones tomadas son:

- **Inconsistencia lógica:** se eliminan 14 registros donde hay camas ocupadas pero camas disponibles igual a cero, lo que es físicamente imposible.
- **Rango teórico:** se eliminan 1.778 registros con INDICE_OCUPACIONAL mayor a 100%, que excede el límite de un porcentaje.
- **Outliers:** no se eliminan porque son datos reales con significado clínico (por ejemplo, psiquiatría puede tener estadías de más de 800 días). La variable COMPLEJIDAD que creamos a continuación controla esta heterogeneidad.

También creamos la variable COMPLEJIDAD, que agrupa las 31 áreas funcionales en tres niveles según el tipo de atención que prestan.

In [ ]:
n_antes = len(df)

# Eliminamos inconsistencias logicas
df = df[~(
    (df['DIAS_CAMAS_DISPONIBLES'] == 0) &
    (df['DIAS_CAMAS_OCUPADAS'] > 0)
)].copy()

# Eliminamos registros con indice ocupacional fuera de rango
df = df[
    (df['INDICE_OCUPACIONAL'] >= 0) &
    (df['INDICE_OCUPACIONAL'] <= 100)
].copy()

# Creamos la variable COMPLEJIDAD clasificando areas funcionales
# Alta complejidad: unidades criticas y de cuidados intensivos
AREAS_ALTA = [
    'Area Cuidados Intensivos Adultos',
    'Area Cuidados Intensivos Pediatricos',
    'Area Cuidados Intermedios Adultos',
    'Area Cuidados Intermedios Pediatricos',
    'Area Neonatologia Cuidados Intensivos'
]
# Baja complejidad: hospitalizacion basica y obstetricia
AREAS_BAJA = [
    'Area Medica Adulto Cuidados Basicos',
    'Area Medico-Quirurgico Cuidados Basicos',
    'Area Obstetricia'
]
# Todo lo demas queda como media complejidad

df['COMPLEJIDAD'] = np.where(
    df['AREA_FUNCIONAL'].isin(AREAS_ALTA), 'Alta complejidad',
    np.where(df['AREA_FUNCIONAL'].isin(AREAS_BAJA), 'Baja complejidad', 'Media complejidad')
)

n_despues = len(df)
print(f"Registros eliminados en limpieza: {n_antes - n_despues:,}")
print(f"Dataset limpio:                   {n_despues:,} filas | {df.shape[1]} columnas")
print(f"\nINDICE_OCUPACIONAL — Min: {df['INDICE_OCUPACIONAL'].min():.2f} | Max: {df['INDICE_OCUPACIONAL'].max():.2f}")
print(f"LETALIDAD          — Min: {df['LETALIDAD'].min():.2f} | Max: {df['LETALIDAD'].max():.2f}")
print(f"\nDistribucion por complejidad:")
print(df['COMPLEJIDAD'].value_counts())

# Verificamos que no queden problemas después de la limpieza
reporte_post = detectar_problemas_datos(df)
problemas_post = reporte_post[reporte_post[['nulos','negativos','fuera_de_rango']].sum(axis=1) > 0]
if problemas_post.empty:
    print("\nDataset limpio sin problemas de calidad.")
else:
    display(problemas_post)

## 7. Análisis Exploratorio de Datos (EDA)

### 7.1 Estadística Descriptiva

Calculamos las medidas de tendencia central y dispersión para cada variable numérica relevante. El foco está en INDICE_OCUPACIONAL y LETALIDAD, las dos variables centrales de la pregunta de investigación.

In [ ]:
# Definimos las variables que vamos a analizar descriptivamente
# Son las mas relevantes para la pregunta de investigacion
indicadores = [
    'INDICE_OCUPACIONAL', 'LETALIDAD', 'PROMEDIO_DIAS_ESTADA',
    'INDICE_ROTACION', 'NUMERO_EGRESOS',
    'DIAS_CAMAS_OCUPADAS', 'DIAS_CAMAS_DISPONIBLES'
]

resumen = []
for col in indicadores:
    # Eliminamos valores nulos antes de calcular para no distorsionar los resultados
    serie = df[col].dropna()
    
    # Calculamos Q1 y Q3 para obtener el rango intercuartilico
    # El IQR es la distancia entre el 25% y el 75% de los datos y sirve para detectar outliers
    Q1  = serie.quantile(0.25)
    Q3  = serie.quantile(0.75)
    IQR = Q3 - Q1
    
    resumen.append({
        'Variable':   col,
        'n':          len(serie),
        'Media':      round(serie.mean(), 2),
        'Mediana':    round(serie.median(), 2),
        # ddof=1 calcula la desviacion estandar muestral, que es la correcta cuando trabajamos con una muestra y no con toda la poblacion
        'Desv. Std':  round(serie.std(ddof=1), 2),
        'Min':        round(serie.min(), 2),
        'Q1':         round(Q1, 2),
        'Q3':         round(Q3, 2),
        'Max':        round(serie.max(), 2),
        'IQR':        round(IQR, 2),
        # La asimetria mide si la distribucion se carga mas hacia la derecha o la izquierda
        # Un valor positivo indica cola a la derecha, negativo indica cola a la izquierda
        'Asimetria':  round(serie.skew(), 3),
    })

df_desc = pd.DataFrame(resumen).set_index('Variable')
display(df_desc)

**Interpretación:** LETALIDAD tiene fuerte asimetría positiva (media 5,1% vs mediana 1,5%), lo que indica que la mayoría de los registros tiene mortalidad baja, pero hay casos extremos con valores muy altos concentrados en áreas de alta complejidad. INDICE_OCUPACIONAL es más simétrico, con media y mediana cercanas al 63%.

### 7.2 Variables Categóricas

In [ ]:
# Frecuencias por nivel de complejidad
print("COMPLEJIDAD — Frecuencias absolutas y relativas")
print("="*50)
comp_freq = df['COMPLEJIDAD'].value_counts()
comp_pct  = df['COMPLEJIDAD'].value_counts(normalize=True).mul(100).round(2)
df_comp   = pd.DataFrame({'Frecuencia': comp_freq, 'Porcentaje (%)': comp_pct})
display(df_comp)

# Top 10 areas funcionales por numero de registros
print("\nAREA FUNCIONAL — Top 10 por frecuencia")
print("="*50)
area_freq = df['AREA_FUNCIONAL'].value_counts().head(10)
area_pct  = df['AREA_FUNCIONAL'].value_counts(normalize=True).mul(100).round(2).head(10)
df_area   = pd.DataFrame({'Frecuencia': area_freq, 'Porcentaje (%)': area_pct})
display(df_area)

**Interpretación:** La media complejidad concentra casi la mitad de los registros (49,1%). La alta complejidad, aunque minoritaria (20,2%), concentra los valores más extremos de letalidad.

### 7.3 Detección de Outliers

Usamos el método IQR para identificar valores atípicos. Un valor es outlier si está por debajo de Q1 - 1.5×IQR o por encima de Q3 + 1.5×IQR. Los outliers no se eliminan porque representan datos clínicamente reales.

In [ ]:
print("DETECCION DE OUTLIERS - Metodo IQR (1.5xIQR)")
print("="*60)

resumen_outliers = []
for col in indicadores:
    serie = df[col].dropna()
    Q1    = serie.quantile(0.25)
    Q3    = serie.quantile(0.75)
    IQR   = Q3 - Q1
    
    # Los limites se calculan como Q1 menos 1.5 veces el IQR por abajo
    # y Q3 mas 1.5 veces el IQR por arriba
    # Todo lo que quede fuera de ese rango se considera outlier
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    
    # Contamos cuantos valores caen fuera de los limites
    n_out   = ((serie < lim_inf) | (serie > lim_sup)).sum()
    pct     = n_out / len(serie) * 100
    
    resumen_outliers.append({
        'Variable':       col,
        'Q1':             round(Q1, 2),
        'Q3':             round(Q3, 2),
        'IQR':            round(IQR, 2),
        'Lim. inferior':  round(lim_inf, 2),
        'Lim. superior':  round(lim_sup, 2),
        'N outliers':     n_out,
        'Porcentaje (%)': round(pct, 2)
    })

df_out = pd.DataFrame(resumen_outliers).set_index('Variable')
display(df_out)

print()
# Interpretamos el resultado de cada variable segun la cantidad de outliers que tiene
for col, row in df_out.iterrows():
    if row['N outliers'] == 0:
        print(f"{col}: sin outliers")
    elif row['Porcentaje (%)'] < 5:
        # Pocos outliers, probablemente casos puntuales que no distorsionan el analisis
        print(f"{col}: {row['N outliers']:,} outliers ({row['Porcentaje (%)']:.2f}%) - se conservan para el analisis")
    else:
        # Muchos outliers indica que la distribucion tiene una cola larga, no son errores sino datos reales
        print(f"{col}: {row['N outliers']:,} outliers ({row['Porcentaje (%)']:.2f}%) - distribucion con cola larga")

**Interpretación:** LETALIDAD y PROMEDIO_DIAS_ESTADA presentan el mayor porcentaje de outliers. Esto es esperable: psiquiatría tiene estadías de cientos de días y las UCIs tienen mortalidades muy distintas al resto. No los eliminamos porque son valores reales que representan segmentos reales del sistema.

## 8. Visualizaciones Exploratorias

### Figura 1 — Histogramas de Variables Numéricas

Visualizamos la distribución de cada variable para identificar su forma, asimetría y comportamiento general.

In [ ]:
# Creamos una grilla de 2 filas y 4 columnas para mostrar un histograma por variable
# Como tenemos 7 variables, el ultimo espacio queda vacio
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Distribucion de Variables Numericas (post-limpieza)', fontsize=13, fontweight='bold')

# flatten convierte la grilla de 2x4 en una lista de 8 ejes, mas facil de recorrer con un for
axes = axes.flatten()

for i, col in enumerate(indicadores):
    serie = df[col].dropna()
    
    # Dibujamos el histograma con 40 barras para ver bien la forma de la distribucion
    axes[i].hist(serie, bins=40, color='steelblue', edgecolor='white')
    
    # Agregamos lineas verticales para la media y la mediana
    # Si ambas lineas estan cerca, la distribucion es simetrica
    # Si estan lejos, hay asimetria y la media esta siendo arrastrada por valores extremos
    axes[i].axvline(serie.mean(),   color='red',    linestyle='--', linewidth=1.5, label='Media')
    axes[i].axvline(serie.median(), color='orange', linestyle='--', linewidth=1.5, label='Mediana')
    
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')
    axes[i].legend(fontsize=7)

# Ocultamos el ultimo subplot que quedo vacio
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

**Interpretación:** INDICE_OCUPACIONAL es aproximadamente simétrico (media y mediana cercanas al 63%), lo que indica que la mayoría de los hospitales opera en rangos similares. LETALIDAD tiene cola pronunciada a la derecha (mediana 1,5% vs media 5,1%), reflejando que la mortalidad alta está concentrada en áreas específicas. La heterogeneidad del sistema hospitalario obliga a analizar las variables separadas por grupos de complejidad.

### Figura 2 — Boxplots por Nivel de Complejidad

Separamos las distribuciones por nivel de complejidad para ver si hay diferencias estructurales entre grupos.

In [ ]:
# Creamos dos boxplots lado a lado: uno para el indice ocupacional y otro para la letalidad
# El objetivo es ver si las distribuciones de ambas variables difieren segun el nivel de complejidad
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Distribucion de Variables Clave por Nivel de Complejidad', fontsize=13, fontweight='bold')

# Fijamos el orden de los grupos para que siempre aparezcan de mayor a menor complejidad
orden = ['Alta complejidad', 'Media complejidad', 'Baja complejidad']

# Usamos zip para recorrer los dos graficos y sus variables al mismo tiempo
# en cada iteracion ax es el subplot, col es la variable y titulo es la etiqueta del eje
for ax, col, titulo in zip(axes,
                            ['INDICE_OCUPACIONAL', 'LETALIDAD'],
                            ['Indice Ocupacional (%)', 'Letalidad (%)']):
    sns.boxplot(
        data=df,
        x='COMPLEJIDAD',
        y=col,
        order=orden,
        palette={'Alta complejidad': 'tomato',
                 'Media complejidad': 'steelblue',
                 'Baja complejidad': 'seagreen'},
        ax=ax,
        # fliersize controla el tamaño de los puntos que representan outliers
        # lo ponemos pequeno para que no saturen el grafico
        fliersize=2
    )
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Nivel de complejidad')
    ax.set_ylabel(titulo)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

**Interpretación:** La alta complejidad lidera con IO mediana del 85% (justo en el umbral crítico de la literatura) y letalidad mediana del 20%. Media y baja se mantienen entre 60-70% de IO y menos del 10% de letalidad. Tanto la ocupación como la mortalidad son significativamente más altos en alta complejidad, lo que sugiere una posible relación entre ambas variables.

## 9. Análisis de Relaciones entre Variables

Esta sección explora la relación entre ocupación y letalidad, que es el núcleo de la pregunta de investigación. Analizamos tres dimensiones: relación global, relación por complejidad y relación por rangos de ocupación.

In [ ]:
# Filtramos registros con actividad hospitalaria registrada para el analisis de P5
# Excluimos registros con IO = 0 porque representan meses sin actividad, no hospitales vacios
df_p5 = df[(df['INDICE_OCUPACIONAL'] > 0) & (df['LETALIDAD'] >= 0)].copy()

print(f"Registros para analisis: {len(df_p5):,}")
print()

# Calculamos correlacion global entre las dos variables centrales
r_pearson,  p_pearson  = pearsonr(df_p5['INDICE_OCUPACIONAL'], df_p5['LETALIDAD'])
r_spearman, p_spearman = spearmanr(df_p5['INDICE_OCUPACIONAL'], df_p5['LETALIDAD'])

print("CORRELACION GLOBAL")
print("="*55)
print(f"Pearson:  r = {r_pearson:.4f}  (p-value = {p_pearson:.4e})")
print(f"Spearman: r = {r_spearman:.4f}  (p-value = {p_spearman:.4e})")
print()

# La correlacion por grupo nos permite ver si la relacion varia segun complejidad
print("CORRELACION POR NIVEL DE COMPLEJIDAD (Pearson)")
print("="*55)
for comp in ['Alta complejidad', 'Media complejidad', 'Baja complejidad']:
    sub = df_p5[df_p5['COMPLEJIDAD'] == comp]
    r, p = pearsonr(sub['INDICE_OCUPACIONAL'], sub['LETALIDAD'])
    print(f"  {comp:<25} r = {r:.4f}  (p = {p:.4e},  n = {len(sub):,})")

### Figura 3 — Dispersión entre Índice Ocupacional y Letalidad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Relacion entre Indice Ocupacional y Letalidad', fontsize=14, fontweight='bold')

# Tomamos una muestra para no saturar el grafico, pero la linea de regresion usa todos los datos
sample = df_p5.sample(min(50000, len(df_p5)), random_state=42)

ax = axes[0]
ax.scatter(sample['INDICE_OCUPACIONAL'], sample['LETALIDAD'],
           alpha=0.05, s=3, color='steelblue')

m, b = np.polyfit(df_p5['INDICE_OCUPACIONAL'], df_p5['LETALIDAD'], 1)
x_line = np.linspace(0, 100, 200)
ax.plot(x_line, m * x_line + b, color='red', linewidth=2,
        label=f'Regresion (r = {r_pearson:.3f})')

ax.set_xlabel('Indice Ocupacional (%)')
ax.set_ylabel('Letalidad (%)')
ax.set_title('Dispersion Global')
ax.set_xlim(0, 105)
ax.set_ylim(0, 105)
ax.legend(fontsize=9)

ax = axes[1]
for comp in ['Alta complejidad', 'Media complejidad', 'Baja complejidad']:
    sub = df_p5[df_p5['COMPLEJIDAD'] == comp].sample(
        min(15000, len(df_p5[df_p5['COMPLEJIDAD'] == comp])), random_state=42)
    ax.scatter(sub['INDICE_OCUPACIONAL'], sub['LETALIDAD'],
               alpha=0.08, s=3, color=palette[comp], label=comp)

ax.set_xlabel('Indice Ocupacional (%)')
ax.set_ylabel('Letalidad (%)')
ax.set_title('Dispersion por Nivel de Complejidad')
ax.set_xlim(0, 105)
ax.set_ylim(0, 105)
ax.legend(fontsize=9, markerscale=4)

plt.tight_layout()
plt.show()

**Interpretación:** La tendencia positiva débil (r = 0,226) confirma que a mayor ocupación, levemente mayor mortalidad. La nube dispersa indica que la ocupación sola no explica la letalidad. Al separar por complejidad, las áreas de alta complejidad concentran los valores más altos de mortalidad independientemente del índice ocupacional, confirmando que la complejidad también incide y debe controlarse en el análisis.

### Figura 4 — Letalidad según Rangos de Ocupación

Dividimos el índice ocupacional en cuatro rangos para detectar si existe un umbral a partir del cual la mortalidad sube de forma más pronunciada.

In [ ]:
# Dividimos el indice ocupacional en cuatro rangos para ver si existe un umbral
# a partir del cual la letalidad empieza a subir de forma mas pronunciada
df_p5['RANGO_OCUPACION'] = pd.cut(
    df_p5['INDICE_OCUPACIONAL'],
    bins=[0, 50, 70, 85, 100],
    labels=['0-50% (Baja)', '50-70% (Moderada)', '70-85% (Alta)', '85-100% (Muy alta)']
)

# Calculamos la letalidad promedio, mediana y cantidad de registros por cada rango
p5_bins = (
    df_p5.groupby('RANGO_OCUPACION', observed=True)['LETALIDAD']
    .agg(['mean', 'median', 'count'])
    .reset_index()
)
p5_bins.columns = ['Rango Ocupacion', 'Letalidad Media', 'Letalidad Mediana', 'N registros']

display(p5_bins.round(2))

fig, ax = plt.subplots(figsize=(10, 5))

# Usamos colores que van del verde al rojo para reflejar visualmente el nivel de riesgo
colores_rango = ['seagreen', 'yellowgreen', 'orange', 'tomato']
x = range(len(p5_bins))

bars = ax.bar(x, p5_bins['Letalidad Media'], color=colores_rango, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(p5_bins['Rango Ocupacion'], rotation=15, ha='right')
ax.set_ylabel('Letalidad promedio (%)')
ax.set_title('Letalidad Promedio segun Rango de Ocupacion Hospitalaria', fontweight='bold')

# Agregamos etiquetas encima de cada barra con el porcentaje y el numero de registros
# para que se pueda leer directamente sin necesidad de ir al eje
for bar, row in zip(bars, p5_bins.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            f'{row._2:.2f}%\n(n={row._4:,})',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

**Interpretación:** La letalidad aumenta progresivamente con la ocupación: 2% en rangos bajos, 4,1% en moderada, 5,59% en alta y 9,23% en muy alta. A partir del 70% el aumento se acelera, confirmando la existencia de un efecto umbral: cuando el sistema opera cerca de su límite, los resultados clínicos se deterioran más.

### Figura 5 — Evolución Temporal

In [ ]:
# Calculamos el promedio anual de ambas variables para ver su evolucion en el tiempo
p5_anual = df_p5.groupby('PERIODO').agg(
    INDICE_OCUPACIONAL=('INDICE_OCUPACIONAL', 'mean'),
    LETALIDAD=('LETALIDAD', 'mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))

# Creamos un segundo eje Y sobre el mismo grafico porque las dos variables tienen escalas distintas
# el indice ocupacional va de 0 a 100 y la letalidad va de 0 a valores mucho menores
# si las graficamos en el mismo eje una de las dos quedaria aplastada y no se veria bien
ax2 = ax1.twinx()

ax1.plot(p5_anual['PERIODO'], p5_anual['INDICE_OCUPACIONAL'],
         marker='o', color='steelblue', linewidth=2, label='Indice Ocupacional')
ax2.plot(p5_anual['PERIODO'], p5_anual['LETALIDAD'],
         marker='s', color='tomato', linewidth=2, linestyle='--', label='Letalidad')

ax1.set_xlabel('Ano')
ax1.set_ylabel('Indice Ocupacional (%)', color='steelblue')
ax2.set_ylabel('Letalidad promedio (%)', color='tomato')

# Coloreamos las etiquetas de cada eje para que coincidan con su linea
ax1.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='tomato')

# Como tenemos dos ejes distintos, juntamos las leyendas manualmente en un solo bloque
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

ax1.set_title('Evolucion Temporal del Indice Ocupacional y la Letalidad (2014-2025)',
              fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación:** De 2014 a 2019 ambas variables se mantienen estables. En 2020 el índice ocupacional cae al 55% pero la letalidad sube al 8,7%, lo que refleja el impacto del COVID-19: el sistema se concentró en casos graves y dejó de atender casos leves. A partir de 2021 la ocupación se recupera y supera niveles prepandemia en 2023. Esta figura confirma que la relación entre ocupación y letalidad no es directa: el tipo de paciente atendido puede invertir la tendencia esperada.

### Figura 6 — Matriz de Correlación

In [ ]:
# Seleccionamos las variables mas relevantes para analizar sus correlaciones entre si
# no incluimos todas las del dataset para que la matriz sea legible y facil de interpretar
vars_corr = [
    'INDICE_OCUPACIONAL', 'LETALIDAD', 'PROMEDIO_DIAS_ESTADA',
    'INDICE_ROTACION', 'NUMERO_EGRESOS'
]

# Calculamos la matriz de correlacion de Pearson
# cada celda muestra que tan relacionadas estan dos variables entre si
# 1 es correlacion perfecta positiva, -1 es perfecta negativa y 0 es ninguna relacion
corr_matrix = df_p5[vars_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.3f',
    # coolwarm va de azul (correlacion negativa) a rojo (correlacion positiva)
    # center=0 hace que el blanco quede en cero, facilitando la lectura visual
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor='white',
    annot_kws={'size': 11},
    ax=ax
)
ax.set_title('Matriz de Correlacion de Pearson - Variables Numericas Relevantes',
             fontweight='bold', fontsize=12)
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.tight_layout()
plt.show()

# Mostramos solo las correlaciones con LETALIDAD ordenadas de mayor a menor por valor absoluto
# esto nos permite identificar rapidamente que variables tienen mayor relacion con la mortalidad
# y cuales usar en el modelo de regresion
corr_letalidad = corr_matrix['LETALIDAD'].drop('LETALIDAD').sort_values(key=abs, ascending=False)
print("\nCorrelaciones con LETALIDAD (ordenadas por valor absoluto):")
print("="*50)
for var, val in corr_letalidad.items():
    print(f"  {var:<30} r = {val:.4f}")

**Interpretación:** La correlación más alta con LETALIDAD es la de INDICE_OCUPACIONAL (0,226). PROMEDIO_DIAS_ESTADA tiene correlación casi nula (0,065). INDICE_ROTACION tiene correlación negativa (-0,240): más rotación implica menos mortalidad, probablemente porque esas áreas atienden casos menos graves. Esta matriz orienta la selección de variables para el modelo de regresión.

## 10. Test de Hipótesis

### 10.1 Formulación

Para responder formalmente la pregunta de investigación aplicamos dos tests complementarios:

**Test 1 — Correlación de Pearson**
Evalúa si existe asociación lineal significativa entre INDICE_OCUPACIONAL y LETALIDAD.
- H0: no existe correlación lineal (rho = 0)
- H1: existe correlación lineal significativa (rho distinto de 0)
- Nivel de significancia: alpha = 0,05

**Test 2 — ANOVA de un factor**
Evalúa si la letalidad promedio difiere entre los tres niveles de complejidad.
- H0: las medias son iguales en los tres grupos
- H1: al menos un grupo tiene media distinta
- Nivel de significancia: alpha = 0,05

### 10.2 Justificación

Pearson es el test adecuado porque ambas variables son cuantitativas continuas y queremos evaluar una asociación lineal. El ANOVA corresponde porque comparamos una variable cuantitativa entre más de dos grupos independientes. Con n mayor a 150.000 registros, el Teorema Central del Límite garantiza que los resultados son robustos aunque los datos no distribuyan normal.

### 10.3 Resultados

In [ ]:
print("TEST 1 - CORRELACION DE PEARSON")
print("="*55)
print("H0: rho = 0  (no existe correlacion lineal)")
print("H1: rho != 0  (existe correlacion lineal significativa)")
print(f"Nivel de significancia: alpha = 0.05")
print()

# Calculamos el coeficiente de Pearson y su valor p
# r mide la fuerza y direccion de la relacion lineal entre ambas variables
# p indica si esa relacion es estadisticamente significativa o podria ser producto del azar
r, p = pearsonr(df_p5['INDICE_OCUPACIONAL'], df_p5['LETALIDAD'])

print(f"Coeficiente de Pearson: r = {r:.4f}")
print(f"Valor p:                p = {p:.4e}")
print(f"N:                      {len(df_p5):,}")
print()

# Si el valor p es menor a 0.05 rechazamos la hipotesis nula
# Eso significa que la correlacion que encontramos no es producto del azar
# con mas de 150 mil datos, el Teorema Central del Limite garantiza que el resultado es robusto
# aunque los datos no distribuyan normal
if p < 0.05:
    print("DECISION: Se rechaza H0")
    print(f"   Dado que p = {p:.4e} < 0.05, existe evidencia estadisticamente")
    print(f"   significativa de una correlacion lineal positiva (r = {r:.4f})")
    print(f"   entre el indice ocupacional y la letalidad.")
else:
    print("DECISION: No se rechaza H0")
    print("   No existe evidencia suficiente de correlacion lineal.")

**Interpretación Test 1:** Dado que el valor p es menor a 0,05, se rechaza la hipótesis nula. Existe una correlación positiva significativa (r = 0,226) entre el índice ocupacional y la letalidad. Si bien la relación es débil, el tamaño muestral descarta que sea producto del azar.

In [ ]:
print("TEST 2 - ANOVA DE UN FACTOR")
print("="*55)
print("H0: media_alta = media_media = media_baja")
print("H1: Al menos un grupo tiene media distinta")
print(f"Nivel de significancia: alpha = 0.05")
print()

# Separamos la letalidad en tres listas, una por nivel de complejidad
# el ANOVA necesita los grupos como listas independientes para compararlos
grupos_anova = [
    df_p5[df_p5['COMPLEJIDAD'] == g]['LETALIDAD'].dropna()
    for g in ['Alta complejidad', 'Media complejidad', 'Baja complejidad']
]

# El asterisco desempaqueta la lista de grupos para pasarlos como argumentos separados
f_stat, p_anova = scipy_stats.f_oneway(*grupos_anova)

# El estadistico F mide cuanto mayor es la variabilidad entre grupos comparada con la variabilidad dentro de cada grupo
# Un F muy alto como el que obtenemos indica que las diferencias entre grupos son enormes
print(f"Estadistico F:  {f_stat:.4f}")
print(f"Valor p:        {p_anova:.4e}")
print()

# Mostramos las medias de cada grupo para dimensionar que tan distintos son entre si
print("Letalidad media por grupo:")
for comp in ['Alta complejidad', 'Media complejidad', 'Baja complejidad']:
    media = df_p5[df_p5['COMPLEJIDAD'] == comp]['LETALIDAD'].mean()
    n     = len(df_p5[df_p5['COMPLEJIDAD'] == comp])
    print(f"  {comp:<25} media = {media:.2f}%  (n = {n:,})")

print()
if p_anova < 0.05:
    print("DECISION: Se rechaza H0")
    print(f"   Dado que p = {p_anova:.4e} < 0.05, existen diferencias estadisticamente")
    print(f"   significativas en la letalidad promedio entre los tres niveles")
    print(f"   de complejidad hospitalaria.")
else:
    print("DECISION: No se rechaza H0")

**Interpretación Test 2:** Dado que el valor p es menor a 0,05, se rechaza la hipótesis nula. Existen diferencias estadísticamente significativas en la letalidad entre los tres niveles de complejidad. El estadístico F de 38.124,5 indica que las diferencias entre grupos son enormemente mayores que las diferencias dentro de cada grupo. La diferencia de 28 puntos porcentuales entre alta (29,9%) y media complejidad (1,8%) confirma que el nivel de complejidad es el factor dominante sobre la mortalidad.

## 11. Modelo de Regresión Lineal Múltiple

### 11.1 Objetivo

Estimar cuánto incide el índice de ocupación sobre la letalidad considerando el efecto de otras variables simultáneamente. No buscamos predecir, sino cuantificar el aporte de cada variable.

### 11.2 Especificación

```
LETALIDAD ~ INDICE_OCUPACIONAL + PROMEDIO_DIAS_ESTADA + INDICE_ROTACION + COMPLEJIDAD
```

La variable COMPLEJIDAD se codifica como dummies usando Baja complejidad como referencia. Los coeficientes de alta y media complejidad se interpretan en comparación con baja.

In [ ]:
# Seleccionamos solo las columnas que necesita el modelo y eliminamos filas con valores faltantes
df_reg = df_p5[['LETALIDAD', 'INDICE_OCUPACIONAL', 'PROMEDIO_DIAS_ESTADA',
                 'INDICE_ROTACION', 'COMPLEJIDAD']].dropna().copy()

# Convertimos COMPLEJIDAD a variables dummy para que el modelo pueda usarla
# Usamos Baja complejidad como referencia (drop_first=False y luego eliminamos manualmente)
dummies = pd.get_dummies(df_reg['COMPLEJIDAD'], drop_first=False).astype(int)
dummies = dummies.drop(columns=['Baja complejidad'])
dummies.columns = ['Alta_complejidad', 'Media_complejidad']

df_reg = pd.concat([df_reg.drop(columns='COMPLEJIDAD'), dummies], axis=1)

# Convertimos todo a float para que statsmodels no tenga problemas con los tipos
df_reg = df_reg.astype(float)

X = df_reg[['INDICE_OCUPACIONAL', 'PROMEDIO_DIAS_ESTADA',
            'INDICE_ROTACION', 'Alta_complejidad', 'Media_complejidad']]
y = df_reg['LETALIDAD']

X_sm = sm.add_constant(X)
modelo = sm.OLS(y, X_sm).fit()

print(modelo.summary())

### 11.3 Resultados e Interpretación de Coeficientes

In [ ]:
tabla_modelo = pd.DataFrame({
    'Coeficiente':    modelo.params.round(4),
    'Error Estandar': modelo.bse.round(4),
    'Valor t':        modelo.tvalues.round(4),
    'Valor p':        modelo.pvalues.round(4),
    'IC inf (95%)':   modelo.conf_int()[0].round(4),
    'IC sup (95%)':   modelo.conf_int()[1].round(4),
})
display(tabla_modelo)

print(f"\nR2:          {modelo.rsquared:.4f}")
print(f"R2 ajustado: {modelo.rsquared_adj:.4f}")
print(f"F-statistic: {modelo.fvalue:.2f}  (p = {modelo.f_pvalue:.4e})")
print(f"N:           {int(modelo.nobs):,}")

### 11.4 Evaluación del Modelo

1. INDICE_OCUPACIONAL (b = 0.0907):
   Manteniendo constantes las demas variables, por cada punto porcentual
   que sube la ocupacion, la letalidad sube 0.09 pp en promedio.
   El efecto es pequeno pero estadisticamente significativo (p < 0.001).

2. Alta_complejidad (b = 24.893):
   Las areas de alta complejidad tienen 24.9 pp mas de letalidad que las
   de baja complejidad, manteniendo constantes las demas variables.
   Es el coeficiente mas grande del modelo: la complejidad domina.

3. INDICE_ROTACION (b = -0.533):
   Mayor rotacion de pacientes se asocia con menor letalidad.
   Las areas que rotan mas atienden casos menos graves.

4. PROMEDIO_DIAS_ESTADA (b = 0.004):
   Efecto muy pequeno pero significativo. Hospitalizaciones mas largas
   corresponden a casos mas graves y levemente mas mortalidad.

El modelo explica el 35,6% de la variabilidad de la letalidad (R² = 0,356), razonable para datos operacionales sin información clínica individual. Es globalmente significativo (F = 16.610; p < 0,001) y todos sus coeficientes son significativos.

**Limitaciones:** los residuos no distribuyen normal (asimetría 1,882), lo que sugiere que puede haber relaciones no lineales no capturadas. El Durbin-Watson de 0,958 indica autocorrelación temporal, esperable en datos de panel. El 64,4% restante no explicado sugiere que variables no observadas como dotación de personal, diagnósticos y edad de los pacientes son relevantes.

## 12. Discusión Preliminar

Los tres análisis convergen en la misma conclusión: existe una relación positiva y significativa entre ocupación y letalidad, pero es moderada y está mediada principalmente por el nivel de complejidad del área funcional.

**¿Los resultados apoyan la hipótesis inicial?**
Sí, parcialmente. La correlación (r = 0,226) y el coeficiente de regresión (b = 0,091) confirman que más ocupación implica más mortalidad. Pero el efecto de la complejidad (b = 24,89) es mucho mayor, lo que indica que el tipo de paciente importa más que el nivel de ocupación.

**Hallazgos relevantes:**
- La diferencia de 28 puntos porcentuales entre alta y media complejidad no tiene punto de comparación con ninguna otra variable del modelo.
- Existe un efecto umbral sobre el 70% de ocupación donde la mortalidad sube de forma más pronunciada.
- El año 2020 confirmó que la relación no es lineal: menos ocupación puede implicar más muertes si los pacientes son más graves.

**Hallazgo inesperado:**
La relación negativa entre índice de rotación y letalidad. Las áreas que rotan más pacientes tienen menos mortalidad, probablemente porque atienden casos menos graves.

**Limitaciones:**
- Datos agregados por mes y área, sin información clínica individual.
- Residuos no normales: puede haber relaciones no lineales no capturadas.
- No contamos con variables como edad, diagnóstico, dotación de personal ni sexo del paciente.

## 13. Próximos Pasos para el Avance Final

- **Análisis post-hoc (Tukey HSD):** identificar qué pares de grupos de complejidad difieren significativamente entre sí.
- **Transformación logarítmica en LETALIDAD:** corregir la asimetría de los residuos y evaluar si la relación es mejor capturada con un modelo no lineal.
- **Variables adicionales:** incluir servicio de salud regional y período como variables de control.
- **Gráficos de residuos y Q-Q plot:** evaluación formal de supuestos del modelo.
- **Literatura:** conectar los hallazgos con los umbrales de ocupación recomendados por la OCDE.

## 14. Conclusiones Preliminares

1. **Existe relación entre ocupación y letalidad**, pero es débil (r = 0,226). La saturación hospitalaria sí incide en la mortalidad, aunque no es el factor principal.

2. **La complejidad clínica es el factor dominante.** Las áreas de alta complejidad tienen 24,9 puntos porcentuales más de mortalidad que las de baja complejidad, manteniendo todo lo demás constante.

3. **Hay un efecto umbral a partir del 70% de ocupación**, donde la mortalidad empieza a subir de forma más pronunciada.

4. **La pandemia de 2020 demostró que la relación no es lineal.** Menos ocupación puede implicar más muertes si el sistema concentra su atención en pacientes más graves.

5. **El modelo explica el 35,6% de la variabilidad**, lo que es razonable con datos operacionales. El 64,4% restante sugiere que variables no observadas como edad, diagnóstico y dotación de personal son relevantes para el avance final.